# Exercise 18.3: L2 Regularisation (Weight Decay)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-18/03-l2-regularisation-weight-decay.ipynb)

**Reference:** Chapter 18 — Generalisation, Overfitting and Regularisation (`chapters/generalisation-and-regularisation.tex`)  
**Estimated Time:** 20–25 min  

### Learning Objectives
- Implement the $L_2$-regularised MSE loss $L_{\text{reg}}(\mathbf{w}) = \frac{1}{n}\|X\mathbf{w} - \mathbf{y}\|^2 + \lambda\|\mathbf{w}\|^2$ and its exact gradient.
- Train linear models with varying weight-decay strength $\lambda \in \{0, 0.5, 5.0\}$ via gradient descent.
- Verify that larger $\lambda$ shrinks $\|\mathbf{w}\|_2$ and curbs overfitting on noisy collinear data.


In [ ]:
import numpy as np


### Mathematical Background

To curb overfitting (Section 18.3), **$L_2$ regularisation (weight decay)** adds a squared-magnitude penalty to the data loss:

$$L_{\text{reg}}(\mathbf{w}) = \frac{1}{n}\|X\mathbf{w} - \mathbf{y}\|^2 + \lambda\|\mathbf{w}\|^2$$

Its gradient is:

$$\nabla L_{\text{reg}}(\mathbf{w}) = \frac{2}{n}X^T(X\mathbf{w} - \mathbf{y}) + 2\lambda\mathbf{w}$$

In each gradient descent step $\mathbf{w}_{k+1} = (1 - 2\alpha\lambda)\mathbf{w}_k - \frac{2\alpha}{n}X^T(X\mathbf{w}_k - \mathbf{y})$, the factor $(1 - 2\alpha\lambda)$ decays weights slightly toward zero.


In [ ]:
def ridge_loss_and_grad(
    X: np.ndarray,
    y: np.ndarray,
    w: np.ndarray,
    lam: float,
) -> tuple[float, np.ndarray]:
    """Compute L2-regularised MSE loss and its gradient with respect to w.

    Formulas:
        loss = (1/n) * ||X @ w - y||^2 + lam * ||w||^2
        grad = (2/n) * X.T @ (X @ w - y) + 2 * lam * w

    Args:
        X: Feature matrix of shape (n_samples, n_features).
        y: Target vector of shape (n_samples,).
        w: Weight vector of shape (n_features,).
        lam: Non-negative L2 regularisation strength lambda.

    Returns:
        Tuple (loss, grad).
    """
    # TODO: Compute residual r = X @ w - y, then loss and grad
    raise NotImplementedError("Complete this TODO")


def train_ridge_gd(
    X: np.ndarray,
    y: np.ndarray,
    lam: float,
    lr: float = 0.05,
    num_steps: int = 300,
) -> tuple[np.ndarray, list[float]]:
    """Train L2-regularised linear regression from w = 0 using gradient descent.

    Args:
        X: Training feature matrix of shape (n_samples, n_features).
        y: Training targets of shape (n_samples,).
        lam: Non-negative L2 penalty strength.
        lr: Learning rate.
        num_steps: Number of gradient-descent iterations.

    Returns:
        Tuple (w, loss_history) where loss_history has length num_steps.
    """
    # TODO: Initialise w = np.zeros(X.shape[1]) and run num_steps of gradient descent
    raise NotImplementedError("Complete this TODO")


<details>
<summary>Hint</summary>

- In `ridge_loss_and_grad`, `residual = X @ w - y`, `np.mean(residual ** 2)` equals `(1/n) * ||residual||^2`.
- Don't forget the factor of `2` in `2 * lam * w` when differentiating `lam * ||w||^2`.
- In `train_ridge_gd`, start with `w = np.zeros(X.shape[1], dtype=np.float64)` and update `w = w - lr * grad`.

</details>


In [ ]:
rng = np.random.default_rng(42)
n_samples, n_features = 20, 25
X_tr = rng.normal(size=(n_samples, n_features))
w_true = np.array([2.0, -1.5] + [0.0] * (n_features - 2))
y_tr = X_tr @ w_true + 0.8 * rng.normal(size=n_samples)

X_val = rng.normal(size=(100, n_features))
y_val = X_val @ w_true + 0.8 * rng.normal(size=100)

# 1. Numerical gradient check
w_probe = rng.normal(size=n_features)
_, g_ana = ridge_loss_and_grad(X_tr, y_tr, w_probe, lam=0.5)
eps = 1e-6
g_num = np.zeros_like(w_probe)
for j in range(n_features):
    e = np.zeros_like(w_probe)
    e[j] = eps
    l_p, _ = ridge_loss_and_grad(X_tr, y_tr, w_probe + e, lam=0.5)
    l_m, _ = ridge_loss_and_grad(X_tr, y_tr, w_probe - e, lam=0.5)
    g_num[j] = (l_p - l_m) / (2.0 * eps)
assert np.allclose(g_ana, g_num, atol=1e-5)

# 2. Weight norm shrinks monotonically as lambda increases
w0, hist0 = train_ridge_gd(X_tr, y_tr, lam=0.0, lr=0.05, num_steps=300)
w1, _ = train_ridge_gd(X_tr, y_tr, lam=0.5, lr=0.05, num_steps=300)
w2, _ = train_ridge_gd(X_tr, y_tr, lam=5.0, lr=0.05, num_steps=300)
assert hist0[-1] < hist0[0]
assert np.linalg.norm(w2) < np.linalg.norm(w1) < np.linalg.norm(w0)

# 3. Regularisation improves validation MSE over unregularised fit
mse0 = float(np.mean((X_val @ w0 - y_val) ** 2))
mse1 = float(np.mean((X_val @ w1 - y_val) ** 2))
assert mse1 < mse0, f'Expected L2 val MSE ({mse1:.4f}) < unregularised ({mse0:.4f})'

print(f'||w||_2: lam=0 -> {np.linalg.norm(w0):.3f}, lam=0.5 -> {np.linalg.norm(w1):.3f}, lam=5 -> {np.linalg.norm(w2):.3f}')
print('All Chapter 18.3 assertions passed!')


### Think Deeper

1. Why does $L_2$ regularisation shrink large weights more aggressively than small weights?
2. In practice, why is the bias/intercept term usually **excluded** from the $\lambda\|\mathbf{w}\|^2$ penalty?
